# Detector benchmark — run & compare

This notebook drives the `yolo_bench` library rather than shelling out to the CLI.
The two layers are deliberately separate:

* **inference tier** (`yolo_bench.Detector`, `archs/`, `runtimes/`) — reusable Python, importable by other projects.
* **harness tier** (`yolo_bench.orchestrate`, `yolo_bench.plots`, this notebook) — test orchestration and visualisation.

Two practical consequences:

1. Every configuration runs in **its own subprocess**. Running a whole matrix in one interpreter made
   `host_rss_mb` grow monotonically (1.2 → 4.5 GB) purely from accumulation, which silently invalidated
   the memory metric. Isolation costs a process spawn per config and buys trustworthy numbers.
2. Cells are independent, so you can run a cheap probe, then a bigger matrix, without re-running everything.

## 1. Environment

This repo gets cloned onto an **RTX 3070** and a **Jetson Orin Nano** as well as this 3090 box, and the
result CSVs are merged. So every row carries a full fingerprint: CPU model/cores/RAM, GPU name + compute
capability + architecture, OS/kernel, driver, and library versions (TensorRT, ONNX Runtime, OpenVINO, torch).

That matters because the machines differ in ways that change results:

* TensorRT **11.3** here vs **10.16** on the Orin (JetPack 7.2) — different engine builders.
* The CPU backends are dominated by the host CPU: desktop i7-6700K vs Orin's 6-core Cortex-A78AE.
* On Jetson, the **power mode** changes results more than the software stack does.

In [1]:
import sys, glob, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")

import pandas as pd
import plotly.io as pio

# 'notebook_connected' renders inline but loads plotly.js from a CDN, keeping the
# .ipynb small. Plain 'notebook' embeds all ~4.8 MB of plotly.js in every figure.
for _renderer in ("notebook_connected", "notebook", "iframe"):
    try:
        pio.renderers.default = _renderer
        break
    except Exception:
        continue

from yolo_bench import orchestrate, plots
from yolo_bench.env import human_summary

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

print(human_summary())

host      : server  [x86_64]
os        : Ubuntu 26.04.1 LTS (kernel 7.0.0-31-generic)
cpu       : Intel(R) Core(TM) i7-6700K CPU @ 4.00GHz  [4c/8t, 30.7 GB RAM]
gpu       : NVIDIA GeForce RTX 3090 (Ampere, SM 8.6, 23.6 GB)
driver    : 595.84
cuda/cudnn: 13.0 / 9.40.0
libs      : tensorrt 11.3.0.99, onnxruntime 1.30.0, openvino 2026.4.0, torch 2.14.0+cu130 (cuda)
run       : server-20260926T074656Z at 2026-09-26T09:46:56+0200


## 2. Choose an experiment

Named experiments live in `orchestrate.EXPERIMENTS` so a setup is defined once instead of being re-typed
in every cell. Each is a plain dataclass, so any field can be overridden at call time.

In [2]:
pd.DataFrame([
    {
        "experiment": name,
        "description": exp.description,
        "models": ", ".join(exp.models),
        "runtimes": ", ".join(exp.runtimes),
        "precisions": ", ".join(exp.precisions),
        "frames": exp.frames,
    }
    for name, exp in orchestrate.EXPERIMENTS.items()
])

,experiment,description,models,runtimes,precisions,frames
0,smoke,"One model, one runtime — proves the stack runs...",yolo11s,tensorrt,"fp32, fp16",20
1,gpu,"All CUDA backends x both precisions, all models.","yolo11s, yolo11l, yolo26s, yolo26l","tensorrt, ort_trt, ort_cuda, pytorch, openvino","fp32, fp16",100
2,cpu,CPU baselines only (fp32: CPU EPs have no FP16...,"yolo11s, yolo11l, yolo26s, yolo26l","openvino_cpu, ort_cpu, pytorch_cpu",fp32,100
3,headline,"Everything: all runtimes, both precisions, all...","yolo11s, yolo11l, yolo26s, yolo26l","tensorrt, ort_trt, ort_cuda, pytorch, openvino...","fp32, fp16",100
4,precision,FP16 question only: native TRT vs ORT-CUDA vs ...,"yolo11s, yolo11l, yolo26s, yolo26l","tensorrt, ort_cuda, pytorch","fp32, fp16",100
5,small-vs-large,Same-architecture size scaling at the best-kno...,"yolo11s, yolo11l",tensorrt,fp16,100
6,yolo11-vs-yolo26,Head-rewrite question: yolo11 vs yolo26 at mat...,"yolo11s, yolo11l, yolo26s, yolo26l","tensorrt, ort_cuda",fp16,100


Which runtimes actually work here depends on the environment (for example ORT's TensorRT EP needs a
TensorRT major matching the ORT build). The registry knows, so ask it instead of guessing — unsupported
combinations are reported as **skipped** with a reason, never as a fake measurement.

In [3]:
print(__import__("yolo_bench.runtimes", fromlist=["describe"]).describe())

kind           group precisions   available label
tensorrt       cuda  fp32/fp16    yes       TensorRT (native)
ort_trt        cuda  fp32/fp16    yes       ONNX Runtime (TensorRT EP)
ort_cuda       cuda  fp32/fp16    yes       ONNX Runtime (CUDA EP)
pytorch        cuda  fp32/fp16    yes       PyTorch (CUDA)
openvino       cuda  fp32/fp16    yes       OpenVINO (GPU device)
openvino_cpu   cpu   fp32         yes       OpenVINO (CPU)
ort_cpu        cpu   fp32         yes       ONNX Runtime (CPU EP)
pytorch_cpu    cpu   fp32         yes       PyTorch (CPU)

  tensorrt       deserializes a .engine; where the Ampere tensor-core win shows
  ort_trt        needs a TensorRT major matching the ORT build (see README)
  ort_cuda       fp16 is accepted but gains nothing; read the fp32 row
  pytorch        eager; fp16 is *slower* here (latency-bound, see README)
  openvino       does NOT use TensorRT kernels; ~5x slower on NVIDIA
  openvino_cpu   fastest CPU backend measured here
  ort_cpu        CPU

## 3. Run

Start with `smoke` — one model, one runtime, a few frames. It proves the stack works end-to-end in a
couple of seconds before committing to a long run.

A full `headline` matrix (4 models × 8 runtimes × 2 precisions, 100 frames each) takes roughly 20 minutes
here. Results are written to `results/` so later cells read the CSV instead of re-running.

In [4]:
# Cheap end-to-end probe (~seconds). Override frames/warmup for a quick check.
smoke_rows = orchestrate.run_experiment("smoke", frames=20, warmup=5)
smoke_df = pd.DataFrame(smoke_rows)

orchestrate.write_csv(smoke_rows, "results/notebook_smoke.csv")

cols = ["model", "runtime", "precision", "fps", "infer_ms", "e2e_ms",
        "peak_vram_gb", "runtime_version", "status"]
smoke_df[[c for c in cols if c in smoke_df.columns]].style.format(
    {"fps": "{:.1f}", "infer_ms": "{:.2f}", "e2e_ms": "{:.2f}", "peak_vram_gb": "{:.2f}"}
)

[orchestrate] yolo11s   tensorrt      fp32  fps=  131.6 infer=   4.33ms vram=0.62GB (0.82s)
[orchestrate] yolo11s   tensorrt      fp16  fps=  167.8 infer=   2.73ms vram=0.55GB (0.64s)


,model,runtime,precision,fps,infer_ms,e2e_ms,peak_vram_gb,runtime_version,status
0,yolo11s,tensorrt,fp32,131.6,4.33,7.60,0.62,11.3.0.99,ok
1,yolo11s,tensorrt,fp16,167.8,2.73,5.96,0.55,11.3.0.99,ok


In [5]:
# The main matrix. Change the experiment name (or drop --experiment style args)
# to target a specific question, e.g. "precision", "small-vs-large", "cpu".
#
# runtimes = ["tensorrt", "ort_cuda", "pytorch", "openvino", "ort_cpu"]
# rows = orchestrate.run(
#     models=["yolo11s", "yolo11l", "yolo26s", "yolo26l"],
#     runtimes=runtimes, precisions=["fp32", "fp16"],
#     video="data/sample_1080p_h264.mp4", frames=100, warmup=50,
# )
# orchestrate.write_csv(rows, "results/notebook_run.csv")

# Load a previously saved matrix instead of re-measuring.
df = orchestrate.read_csv("results/final.csv")
print(f"loaded {len(df)} rows")
df[["model", "runtime", "precision", "fps", "infer_ms", "e2e_ms",
    "peak_vram_gb", "runtime_version", "status"]].head(10)

loaded 56 rows


,model,runtime,precision,fps,infer_ms,e2e_ms,peak_vram_gb,runtime_version,status
0,yolo11s,pytorch,fp32,81.828039,9.212484,12.220750,0.553101,2.14.0+cu130,ok
1,yolo11s,ort_cuda,fp32,133.932385,4.496616,7.466454,0.592163,1.30.0,ok
2,yolo11s,tensorrt,fp32,135.723016,4.139613,7.367947,0.621460,11.3.0.99,ok
3,yolo11s,openvino,fp32,32.355023,28.019369,30.907103,0.806335,2026.4.0-22959-99c81491cc3-releases/2026/4,ok
4,yolo11s,pytorch_cpu,fp32,7.212561,135.658406,138.647009,0.320679,2.14.0+cu130,ok
5,yolo11s,ort_cpu,fp32,12.668881,75.757272,78.933567,0.320679,1.30.0,ok
6,yolo11s,openvino_cpu,fp32,11.888797,81.050557,84.112796,0.320679,2026.4.0-22959-99c81491cc3-releases/2026/4,ok
7,yolo11s,pytorch,fp16,71.664747,10.952789,13.953862,0.496460,2.14.0+cu130,ok
8,yolo11s,ort_cuda,fp16,131.372960,4.523804,7.611916,0.592163,1.30.0,ok
9,yolo11s,tensorrt,fp16,167.106642,2.675741,5.984203,0.549194,11.3.0.99,ok


### Machine under test

In [6]:
# Printed as the report header too, so a saved figure is self-describing.
from IPython.display import Markdown
Markdown(plots.machine_header(df))

```
host      : server  [x86_64]
os        : Ubuntu 26.04.1 LTS (kernel 7.0.0-31-generic)
cpu       : Intel(R) Core(TM) i7-6700K CPU @ 4.00GHz  [4c/8t, 30.7 GB RAM]
gpu       : NVIDIA GeForce RTX 3090 (Ampere, SM 8.6, 23.6 GB)
driver    : 595.84
cuda/cudnn: 13.0 / 9.40.0
libs      : tensorrt 11.3.0.99, onnxruntime 1.30.0, openvino 2026.4.0, torch 2.14.0+cu130 (cuda)
run       : server-20260926T064716Z at 2026-09-26T08:47:16+0200
```

## 4. fps — nested FP32/FP16 bars

One subplot per runtime; within each, the four models side by side, each model a **pair** of bars:

* **outer (translucent) bar** = FP32
* **inner (solid, outlined) bar** = FP16, drawn on top

**How to read it:** the visible sliver of outer bar *around* the inner bar is the FP16 speedup. A big frame
of colour means a big win; a cell where the inner bar fills the outer one means FP16 did **not** help.
That is the point of nesting rather than grouping — a regression and a tiny gain look different here,
whereas in a grouped chart they look identical.

Colour encodes the **model**, kept consistent across every figure in this notebook.

In [7]:
fig = plots.fps_nested_bars(df)
fig.show()

Saving is worth doing: the HTML is self-contained and can be attached to a PR or a report.

In [8]:
# plots.write_html(fig, "results/fps_nested.html")

## 5. Other views

Nested bars answer *"did FP16 help?"*. They are a poor answer to *"which runtime wins for this model?"*,
because the inset hides part of the bar. So there is a conventional grouped chart for that question, and
a relative chart for comparing across the 100× range between CUDA and CPU backends.

In [9]:
plots.fps_grouped_bars(df).show()

In [10]:
# Speedup relative to a reference runtime. A linear axis would flatten every
# CPU bar to invisibility, so this is log-scaled.
plots.speedup_vs_reference(df, reference="tensorrt", precision="fp16").show()

CPU backends sit ~20× below the CUDA ones, so a single linear axis needs a log scale to show both.

In [11]:
plots.fps_nested_bars(df, runtimes=["ort_cpu", "openvino_cpu", "pytorch_cpu"], log_y=True).show()

## 6. Cross-machine comparison

Copy the `results/*.csv` files from the 3090, the 3070 and the Orin Nano into `results/` and load them
together. Because each row carries `hostname`, `gpu_name`, `gpu_arch`, `cpu` and the library versions,
rows can be grouped by machine instead of silently averaged together.

Keep these caveats in mind when reading a merged table:

* Engines are **not portable** — the 3070/Orin must build their own `.engine` files.
* The Orin runs TensorRT 10.16, so its `tensorrt` rows are not directly comparable to the 3090's TRT 11.3 rows.
* On the Orin, record the power mode (`nvpmodel`); it dominates the result.

In [12]:
# Merge every results CSV present, tagging the source file.
frames = []
for path in sorted(glob.glob("results/*.csv")):
    part = orchestrate.read_csv(path)
    part["source"] = Path(path).name
    frames.append(part)

if frames:
    allruns = pd.concat(frames, ignore_index=True)
    ok = allruns[allruns["status"] == "ok"]
    print(f"{len(frames)} file(s), {len(ok)} successful rows")
    display(
        ok.groupby(["hostname", "gpu_name", "gpu_arch", "cpu"], dropna=False)["fps"]
          .agg(["count", "max", "median"]).sort_values("max", ascending=False)
    )
else:
    print("no results/*.csv yet — run the cells above, or copy CSVs from another machine")

4 file(s), 48 successful rows


,,,,count,max,median
hostname,gpu_name,gpu_arch,cpu,,,
server,NVIDIA GeForce RTX 3090,Ampere,Intel(R) Core(TM) i7-6700K CPU @ 4.00GHz,46,168.185379,50.472653
NaN,NVIDIA GeForce RTX 3090,NaN,NaN,2,162.777426,145.685214


In [13]:
# Same model/runtime across machines — the actual portability question.
if frames:
    ok = allruns[allruns["status"] == "ok"]
    pivot = ok.pivot_table(
        index=["model", "runtime", "precision"],
        columns="gpu_name", values="fps", aggfunc="median",
    )
    display(pivot.round(1))

gpu_name                        NVIDIA GeForce RTX 3090
model   runtime      precision                         
yolo11l openvino     fp16                           9.4
                     fp32                           9.3
        openvino_cpu fp32                           3.4
        ort_cpu      fp32                           3.7
        ort_cuda     fp16                          78.0
                     fp32                          77.1
        pytorch      fp16                          44.2
                     fp32                          52.4
        pytorch_cpu  fp32                           2.5
        tensorrt     fp16                         136.8
                     fp32                          83.6
yolo11s openvino     fp16                          32.4
                     fp32                          32.4
        openvino_cpu fp32                          11.9
        ort_cpu      fp32                          12.7
        ort_cuda     fp16                         131.4
                     fp32                         133.9
        pytorch      fp16                          71.7
                     fp32                          81.8
        pytorch_cpu  fp32                           7.2
        tensorrt     fp16                         167.1
                     fp32                         131.6
yolo26l openvino     fp16                           9.4
                     fp32                           9.3
        openvino_cpu fp32                           3.5
        ort_cpu      fp32                           3.8
        ort_cuda     fp16                          80.3
                     fp32                          79.3
        pytorch      fp16                          40.6
                     fp32                          48.6
        pytorch_cpu  fp32                           2.4
        tensorrt     fp16                         138.4
                     fp32                          85.0
yolo26s openvino     fp16                          32.5
                     fp32                          32.7
        openvino_cpu fp32                          13.1
        ort_cpu      fp32                          13.3
        ort_cuda     fp16                         131.2
                     fp32                         129.7
        pytorch      fp16                          56.4
                     fp32                          65.6
        pytorch_cpu  fp32                           7.0
        tensorrt     fp16                         168.2
                     fp32                         136.2